# SpectraShift Week 9: seed 43 diagnostics
Use T4 x2 with Internet off and GPU 0. Attach source v8, Week 2 frozen, Week 5 contracts, Week 9 contracts, the matching Week 5 seed dataset, and matching Week 8 evaluation dataset.


In [ ]:
from pathlib import Path
import hashlib, json, os, shutil, sys, yaml

INPUT = Path('/kaggle/input')
projects = [p.parent for p in INPUT.rglob('pyproject.toml') if (p.parent / 'src/spectrashift/train/week9.py').is_file()]
if not projects:
    bundles = sorted(INPUT.rglob('spectrashift-kaggle-source.zip'))
    assert len(bundles) == 1, f'Expected one Week 9 source bundle, found {bundles}'
    source_work = Path('/tmp/spectrashift-week9-source')
    if source_work.exists(): shutil.rmtree(source_work)
    shutil.unpack_archive(str(bundles[0]), str(source_work))
    projects = [source_work]
assert projects, 'No Week 9 source tree found'
PROJECT = sorted(projects, key=lambda path: len(str(path)))[0]
sys.path.insert(0, str(PROJECT / 'src'))
os.chdir(PROJECT)

def unique_file(name):
    candidates = sorted(INPUT.rglob(name))
    by_hash = {}
    for path in candidates:
        by_hash.setdefault(hashlib.sha256(path.read_bytes()).hexdigest(), path)
    assert len(by_hash) == 1, f'Expected one unique {name}; found {candidates}'
    return next(iter(by_hash.values()))

SEED = 43
WORK = Path('/kaggle/working/spectrashift-week9-diagnostics-seed43')
WORK.mkdir(parents=True, exist_ok=True)
MANIFEST = unique_file('partitions.parquet')
NORMALIZATION = unique_file('normalization.json')
STAGED = next(path.parent for path in INPUT.rglob('staging_summary.json'))
WEEK5_CONTRACTS = unique_file('week5_contracts_summary.json')
WEEK9_CONTRACTS = unique_file('week9_contracts_summary.json')
WEEK9_CONTRACT = unique_file('week9_contract.json')
DIAGNOSTIC_LEDGER = unique_file('week9_diagnostic_ledger.csv')
EVAL_LABELS = unique_file('evaluation_labels.parquet')
SUPPORT_CONTRACT = unique_file('support_contract.json')
config = yaml.safe_load((PROJECT / 'configs/analysis/week9.yaml').read_text())
config['paths'].update({
    'manifest_path': str(MANIFEST), 'staged_root': str(STAGED),
    'normalization_path': str(NORMALIZATION), 'week5_contracts_path': str(WEEK5_CONTRACTS),
    'week9_contracts_summary_path': str(WEEK9_CONTRACTS),
    'week9_contract_path': str(WEEK9_CONTRACT), 'diagnostic_ledger_path': str(DIAGNOSTIC_LEDGER),
    'evaluation_labels_path': str(EVAL_LABELS), 'support_contract_path': str(SUPPORT_CONTRACT),
})
RUNTIME_CONFIG = WORK / 'week9.yaml'
RUNTIME_CONFIG.write_text(yaml.safe_dump(config, sort_keys=False))


In [ ]:
import torch
assert torch.cuda.is_available() and 'T4' in torch.cuda.get_device_name(0), 'Select GPU T4 x2'
from spectrashift.train.week9 import run_week9_diagnostics
summary = run_week9_diagnostics(RUNTIME_CONFIG, SEED, [INPUT], WORK)
print(json.dumps({key: value for key, value in summary.items() if key != 'runs'}, indent=2))
assert summary['week9_diagnostics_complete']
assert summary['diagnostic_checkpoint_count'] == 2
assert summary['stress_prediction_domain_count'] == 36
assert summary['new_stress_prediction_domain_count'] == 30
assert summary['nearest_neighbor_row_count'] == 1000
assert summary['encoder_updates_during_week9'] is False
